# 2x3 Quantum Dot Array

This notebook extends the same-side 1x2 layout from notebook 09 into reusable 1xN and 2xN constructions.

A `TopBarrierLinearDotArrayDevice` supplies each 1xN row: plungers and barriers both enter from the top. The `TwoDDotArrayDevice` builds two complete rows, rotates only the lower row by 180 degrees, and appends the unrotated row above it. With N = 3, the result contains six quantum dots.

## 0. Setup

In [1]:
import os
import sys
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

REPO_ROOT = next(
    path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / "src").exists() and (path / "notebooks").exists()
)
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from qd_design import (
    CANONICAL_LOLLIPOP_PLUNGER,
    TopBarrierLinearDotArrayDevice,
    TwoDDotArrayDevice,
    build_simulation_layout,
    make_reference_sige_ge_process_stack,
    plot_layout_spec_2d,
    plot_simulation_layout_3d,
    write_nextnano_input_from_template,
)
import nextnanopp_tools as nnt


def print_json(value):
    print(json.dumps(value, indent=2))


print("REPO_ROOT:", REPO_ROOT)

REPO_ROOT: /Users/robertjovanov/code/qpu-design-automation-toolkit


In [2]:
# Set QD_NOTEBOOK_ARTIFACT_ROOT for a read-only/headless validation run.
ARTIFACT_ROOT = Path(
    os.environ.get("QD_NOTEBOOK_ARTIFACT_ROOT", str(REPO_ROOT))
).resolve()
OUTPUT_DIR = ARTIFACT_ROOT / "data" / "gds"
GENERATED_INPUT_DIR = ARTIFACT_ROOT / "configs" / "robert_inputs" / "generated"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_INPUT_DIR.mkdir(parents=True, exist_ok=True)

TEMPLATE_INPUT_PATH = (
    REPO_ROOT / "configs" / "robert_inputs" / "double_qd" / "3d"
    / "Double_Quantum_Dot_3D.in"
)
GDS_PATH = OUTPUT_DIR / "quantum_dot_array_2x3.gds"
SVG_PATH = OUTPUT_DIR / "quantum_dot_array_2x3.svg"
LAYOUT_SPEC_PATH = OUTPUT_DIR / "quantum_dot_array_2x3_layout_spec.json"
SIMULATION_LAYOUT_PATH = OUTPUT_DIR / "quantum_dot_array_2x3_simulation_layout.json"
GENERATED_INPUT_PATH = GENERATED_INPUT_DIR / "Quantum_Dot_Array_2x3.in"

RUNS_DIR = ARTIFACT_ROOT / "runs"
RUN_TAG = "structure_only_2x3_same_side_rows"
RUN_SIMULATION = os.environ.get("QD_RUN_SIMULATION", "0") == "1"
BIAS_INDEX = 0
RUN_VARIABLE_OVERRIDES = {
    "strain": 0,
    "poisson": 0,
    "quantum": 0,
    "quantum_poisson": 0,
}

print("ARTIFACT_ROOT:", ARTIFACT_ROOT)
print("Template exists:", TEMPLATE_INPUT_PATH.exists())
print("RUN_SIMULATION:", RUN_SIMULATION)
print("Structure-only switches:", RUN_VARIABLE_OVERRIDES)

ARTIFACT_ROOT: /Users/robertjovanov/code/qpu-design-automation-toolkit
Template exists: True
RUN_SIMULATION: False
Structure-only switches: {'strain': 0, 'poisson': 0, 'quantum': 0, 'quantum_poisson': 0}


## 1. Build a 1x3 row and compose the 2x3 array

In [3]:
print_json(CANONICAL_LOLLIPOP_PLUNGER.to_dict())

common_row_parameters = dict(
    device_y_size_nm=200.0,
    ohmic_width_nm=40.0,
    ohmic_length_nm=200.0,
    barrier_width_nm=40.0,
    barrier_length_nm=140.0,
    ohmic_to_barrier_gap_nm=20.0,
    barrier_to_plunger_gap_nm=20.0,
)

# Reusable 1xN primitive: barriers and plungers both enter from the top.
one_by_three = TopBarrierLinearDotArrayDevice(
    name="same_side_1x3_source",
    n_dots=3,
    **common_row_parameters,
)

# Reusable 2xN composition: two 1xN builders, with only the lower row rotated.
two_d_array = TwoDDotArrayDevice(
    name="quantum_dot_array_2x3",
    n_columns=3,
    row_gap_nm=0.0,
    **common_row_parameters,
)

one_by_three.ensure_built()
layout = two_d_array.ensure_built()
layout_spec = two_d_array.layout_spec()

print_json(two_d_array.summary())
print("Layout bbox:", layout.bbox)

{
  "body_width_nm": 40.0,
  "body_length_nm": 50.0,
  "head_top_width_nm": 60.0,
  "head_max_width_nm": 100.0,
  "head_height_nm": 100.0,
  "upper_taper_height_nm": 25.0,
  "lower_taper_height_nm": 25.0
}
{
  "name": "quantum_dot_array_2x3",
  "n_rows": 2,
  "n_columns": 3,
  "n_dots": 6,
  "row_builder": "TopBarrierLinearDotArrayDevice",
  "rotated_row": "bottom",
  "device_y_size_nm": 200.0,
  "row_gap_nm": 0.0,
  "row_pitch_y_nm": 200.0,
  "plunger_pitch_x_nm": 180.0,
  "facing_plunger_gap_nm": 100.0,
  "ohmic_width_nm": 40.0,
  "ohmic_length_nm": 200.0,
  "barrier_width_nm": 40.0,
  "barrier_length_nm": 140.0,
  "plunger_body_width_nm": 40.0,
  "plunger_body_length_nm": 50.0,
  "plunger_head_top_width_nm": 60.0,
  "plunger_head_max_width_nm": 100.0,
  "plunger_head_height_nm": 100.0,
  "plunger_upper_taper_height_nm": 25.0,
  "plunger_lower_taper_height_nm": 25.0,
  "ohmic_to_barrier_gap_nm": 20.0,
  "barrier_to_plunger_gap_nm": 20.0,
  "include_screening_gates": false
}
Layout bb

### 1.1 Exact top-down layout

In [4]:
layout_plan_fig = plot_layout_spec_2d(
    layout_spec,
    title="2x3 quantum-dot array: exact PHIDL polygon outlines",
)
layout_plan_fig.show()

### 1.2 Verify counts, transforms, naming, and approach sides

Names are assigned after each row transform, so numbering remains left-to-right even though the 180-degree rotation reverses the lower source row.

In [5]:
elements_by_name = {element["name"]: element for element in layout_spec}
counts = pd.Series(
    [element["gate_type"] for element in layout_spec]
).value_counts().to_dict()

assert counts == {"barrier": 8, "plunger": 6, "ohmic": 4}
assert two_d_array.n_dots == 6
assert len(elements_by_name) == 18
assert len({element["voltage_label"] for element in layout_spec}) == 18
assert two_d_array.row_builders["bottom"] is not two_d_array.row_builders["top"]
assert int(two_d_array.refs["bottom_row"].rotation or 0) % 360 == 180
assert int(two_d_array.refs["top_row"].rotation or 0) % 360 == 0

source_plungers = sorted(
    (
        element for element in one_by_three.layout_spec()
        if element["gate_type"] == "plunger"
    ),
    key=lambda element: element["center_x_nm"],
)
bottom_plungers = [elements_by_name[f"P{index}"] for index in range(1, 4)]
top_plungers = [elements_by_name[f"P{index}"] for index in range(4, 7)]

def transformed_polygon(polygons, *, rotate_180, y_translation_nm):
    result = []
    for polygon in polygons:
        if rotate_180:
            result.append([
                (-x, -y + y_translation_nm)
                for x, y in polygon
            ])
        else:
            result.append([
                (x, y + y_translation_nm)
                for x, y in polygon
            ])
    return result

# Rotation reverses source x order; exported labels are reassigned physically.
for source, target in zip(reversed(source_plungers), bottom_plungers):
    expected = transformed_polygon(
        source["polygon_xy_nm"],
        rotate_180=True,
        y_translation_nm=two_d_array.device_y_size_nm,
    )
    assert np.allclose(target["polygon_xy_nm"], expected)

for source, target in zip(source_plungers, top_plungers):
    expected = transformed_polygon(
        source["polygon_xy_nm"],
        rotate_180=False,
        y_translation_nm=two_d_array.row_pitch_y_nm,
    )
    assert np.allclose(target["polygon_xy_nm"], expected)

bottom_same_side = [
    elements_by_name[name]["y_min_nm"]
    for name in ("P1", "P2", "P3", "B1", "B2", "B3", "B4")
]
top_outer_y_nm = 2.0 * two_d_array.device_y_size_nm + two_d_array.row_gap_nm
top_same_side = [
    elements_by_name[name]["y_max_nm"]
    for name in ("P4", "P5", "P6", "B5", "B6", "B7", "B8")
]
assert bottom_same_side == [0.0] * 7
assert top_same_side == [top_outer_y_nm] * 7
assert [
    [len(polygon) for polygon in element["polygon_xy_nm"]]
    for element in layout_spec
    if element["gate_type"] == "plunger"
] == [[10]] * 6

display(pd.DataFrame([
    {"check": "plungers", "value": counts["plunger"], "expected": 6},
    {"check": "barriers", "value": counts["barrier"], "expected": 8},
    {"check": "ohmics", "value": counts["ohmic"], "expected": 4},
    {"check": "bottom row rotation", "value": 180, "expected": 180},
    {"check": "top row rotation", "value": 0, "expected": 0},
]))
print("Verified: two independent 1x3 builders, only the lower row rotated.")
print("Verified: each row keeps its plungers and barriers on the same outer side.")

,check,value,expected
0,plungers,6,6
1,barriers,8,8
2,ohmics,4,4
3,bottom row rotation,180,180
4,top row rotation,0,0


Verified: two independent 1x3 builders, only the lower row rotated.
Verified: each row keeps its plungers and barriers on the same outer side.


In [6]:
two_d_array.write_gds(str(GDS_PATH))
two_d_array.write_svg(str(SVG_PATH))
two_d_array.write_layout_spec_json(str(LAYOUT_SPEC_PATH))

print("GDS:", GDS_PATH)
print("SVG:", SVG_PATH)
print("Layout spec:", LAYOUT_SPEC_PATH)

GDS: /Users/robertjovanov/code/qpu-design-automation-toolkit/data/gds/quantum_dot_array_2x3.gds
SVG: /Users/robertjovanov/code/qpu-design-automation-toolkit/data/gds/quantum_dot_array_2x3.svg
Layout spec: /Users/robertjovanov/code/qpu-design-automation-toolkit/data/gds/quantum_dot_array_2x3_layout_spec.json


## 2. Build and inspect the 3D simulation layout

In [7]:
process_stack = make_reference_sige_ge_process_stack()
simulation_layout = build_simulation_layout(
    name="quantum_dot_array_2x3_3d",
    layout_elements=layout_spec,
    process_stack=process_stack,
    x_margin_nm=0.0,
    y_margin_nm=0.0,
)
simulation_layout.write_json(str(SIMULATION_LAYOUT_PATH))

patterned_summary = pd.DataFrame(
    {
        "name": region.name,
        "gate_type": region.gate_type,
        "polygon_count": len(region.polygon_xy_nm),
        "vertex_counts": [len(polygon) for polygon in region.polygon_xy_nm],
        "z_min_nm": region.z_min_nm,
        "z_max_nm": region.z_max_nm,
    }
    for region in simulation_layout.patterned_regions
)
display(patterned_summary)
assert len(patterned_summary) == 18
print("Simulation layout:", SIMULATION_LAYOUT_PATH)

,name,gate_type,polygon_count,vertex_counts,z_min_nm,z_max_nm
0,OC_L_1,ohmic,1,[4],-77.0,173.0
1,B1,barrier,1,[4],108.0,138.0
2,P1,plunger,1,[10],143.0,173.0
3,B2,barrier,1,[4],108.0,138.0
4,P2,plunger,1,[10],143.0,173.0
5,B3,barrier,1,[4],108.0,138.0
6,P3,plunger,1,[10],143.0,173.0
7,B4,barrier,1,[4],108.0,138.0
8,OC_R_1,ohmic,1,[4],-77.0,173.0
9,OC_L_2,ohmic,1,[4],-77.0,173.0


Simulation layout: /Users/robertjovanov/code/qpu-design-automation-toolkit/data/gds/quantum_dot_array_2x3_simulation_layout.json


### 2.1 Polygon-preserving 3D structure

The shared renderer extrudes the true lollipop outlines rather than their bounding boxes. Set `STRUCTURE_Z_RANGE_NM = None` to include the full substrate depth.

In [8]:
STRUCTURE_Z_RANGE_NM = (-120.0, 180.0)

structure_preview_fig = plot_simulation_layout_3d(
    simulation_layout,
    z_range_nm=STRUCTURE_Z_RANGE_NM,
    show_polygon_outlines=True,
)
structure_preview_fig.show()

## 3. Write and validate the structure-only nextnano input

In [9]:
if not TEMPLATE_INPUT_PATH.exists():
    raise FileNotFoundError(TEMPLATE_INPUT_PATH)

voltage_overrides = {
    element["voltage_label"]: (
        -3.0 if element["gate_type"] == "plunger" else 0.0
    )
    for element in layout_spec
}
assert len(voltage_overrides) == 18

write_nextnano_input_from_template(
    simulation_layout=simulation_layout,
    template_path=TEMPLATE_INPUT_PATH,
    output_path=GENERATED_INPUT_PATH,
    voltage_overrides=voltage_overrides,
)

generated_input = nnt.load_input_file(GENERATED_INPUT_PATH)
nnt.set_input_variables(generated_input, RUN_VARIABLE_OVERRIDES)
nnt.save_input_file(
    generated_input,
    fullpath=GENERATED_INPUT_PATH,
    overwrite=True,
)

generated_text = GENERATED_INPUT_PATH.read_text(encoding="utf-8")
solver_values = {
    name: (
        match.group(1).strip()
        if (
            match := re.search(
                rf"^\${name}\s*=\s*([^#\n]+)",
                generated_text,
                re.MULTILINE,
            )
        )
        else None
    )
    for name in RUN_VARIABLE_OVERRIDES
}
assert all(str(value) == "0" for value in solver_values.values())

print("Generated input:", GENERATED_INPUT_PATH)
print("Voltage variables:", len(voltage_overrides))
print("Solver switches:", solver_values)

Generated input: /Users/robertjovanov/code/qpu-design-automation-toolkit/configs/robert_inputs/generated/Quantum_Dot_Array_2x3.in
Voltage variables: 18
Solver switches: {'strain': '0', 'poisson': '0', 'quantum': '0', 'quantum_poisson': '0'}


In [10]:
def patterned_region_text(input_text, region_name):
    marker = f"# patterned region: {region_name}"
    start = input_text.index(marker)
    next_marker = input_text.find("# patterned region:", start + len(marker))
    return input_text[start:] if next_marker < 0 else input_text[start:next_marker]


export_checks = []
for region in simulation_layout.patterned_regions:
    block = patterned_region_text(generated_text, region.name)
    expected_vertices = sum(len(polygon) for polygon in region.polygon_xy_nm)
    written_vertices = block.count("vertex{")
    export_checks.append(
        {
            "name": region.name,
            "gate_type": region.gate_type,
            "expected_vertices": expected_vertices,
            "written_vertices": written_vertices,
            "uses_polygonal_prism": "polygonal_prism{" in block,
        }
    )

export_checks = pd.DataFrame(export_checks)
display(export_checks)
assert (export_checks["expected_vertices"] == export_checks["written_vertices"]).all()
assert export_checks["uses_polygonal_prism"].all()
assert export_checks.loc[
    export_checks["gate_type"] == "plunger", "written_vertices"
].tolist() == [10] * 6

print("Verified: nextnano receives six 10-vertex lollipop plunger prisms.")

,name,gate_type,expected_vertices,written_vertices,uses_polygonal_prism
0,OC_L_1,ohmic,4,4,True
1,B1,barrier,4,4,True
2,P1,plunger,10,10,True
3,B2,barrier,4,4,True
4,P2,plunger,10,10,True
5,B3,barrier,4,4,True
6,P3,plunger,10,10,True
7,B4,barrier,4,4,True
8,OC_R_1,ohmic,4,4,True
9,OC_L_2,ohmic,4,4,True


Verified: nextnano receives six 10-vertex lollipop plunger prisms.


## 4. Optionally run nextnano in structure-only mode

The simulation remains off by default. Set `QD_RUN_SIMULATION=1` before starting the kernel, or set `RUN_SIMULATION = True` explicitly, to launch it.

In [12]:
RUN_SIMULATION = True

In [13]:
if RUN_SIMULATION:
    RUNS_DIR.mkdir(parents=True, exist_ok=True)
    input_file = nnt.run_input_file(
        GENERATED_INPUT_PATH,
        output_root=RUNS_DIR,
        tag=RUN_TAG,
        add_timestamp=True,
        variables=RUN_VARIABLE_OVERRIDES,
        show_log=True,
        convergenceCheck=False,
        staging_root=GENERATED_INPUT_DIR / "staged_runs",
        keep_staged_input=True,
    )
    OUTPUT_RUN_DIR = nnt.get_output_directory(input_file)
    print("Structure-only run completed:", OUTPUT_RUN_DIR)
else:
    OUTPUT_RUN_DIR = None
    print("RUN_SIMULATION is False; no nextnano process was launched.")


PREPARING...

Checking input files...


Reading environment and checking paths...

RUNNING..
Starting execution as:
/Applications/nextnano/2025_12_17/nextnano++/bin/nextnano++_gcc_macOS --old --license /Applications/nextnano/2025_12_17/licenses/license.txt --database /Applications/nextnano/2025_12_17/nextnano++/database/database.nnp --threads 0 --outputdirectory /Users/robertjovanov/code/qpu-design-automation-toolkit/runs/Quantum_Dot_Array_2x3__20260718_224604__structure_only_2x3_same_side_rows --noautooutdir /Users/robertjovanov/code/qpu-design-automation-toolkit/configs/robert_inputs/generated/staged_runs/Quantum_Dot_Array_2x3__20260718_224604__structure_only_2x3_same_side_rows.in 

nextnano++ 2.4.27 from 2025-12-16 (library 2025-07-29)
COPYRIGHT NOTICE                                                             
ANY USE OF THE NEXTNANO++ CODE CONSTITUTES ACCEPTANCE OF THE TERMS OF THE    
COPYRIGHT NOTICE.                                                            
                

In [14]:
RUNS_DIR.mkdir(parents=True, exist_ok=True)
if OUTPUT_RUN_DIR is None:
    matching_runs = nnt.find_runs_for_input(RUNS_DIR, GENERATED_INPUT_PATH)
    tagged_runs = [run for run in matching_runs if RUN_TAG in run.name]
    if tagged_runs:
        matching_runs = tagged_runs
    OUTPUT_RUN_DIR = matching_runs[-1] if matching_runs else None

RUN_AVAILABLE = OUTPUT_RUN_DIR is not None and Path(OUTPUT_RUN_DIR).exists()
if RUN_AVAILABLE:
    OUTPUT_RUN_DIR = nnt.resolve_run_root(OUTPUT_RUN_DIR)
    STRUCTURE_DIR = OUTPUT_RUN_DIR / "Structure"
    print("Selected run:", OUTPUT_RUN_DIR)
    print("Structure directory:", STRUCTURE_DIR)
else:
    STRUCTURE_DIR = None
    print("No matching completed run is available yet.")

Selected run: /Users/robertjovanov/code/qpu-design-automation-toolkit/runs/Quantum_Dot_Array_2x3__20260718_224604__structure_only_2x3_same_side_rows
Structure directory: /Users/robertjovanov/code/qpu-design-automation-toolkit/runs/Quantum_Dot_Array_2x3__20260718_224604__structure_only_2x3_same_side_rows/Structure


## 5. Plot the actual nextnano Structure output

This reads contact indices from nextnano's Structure directory, independently checking the post-export geometry. Contact names and colors are derived from the reusable layout rather than hard-coded for N = 3.

In [15]:
VTR_DATA_ARRAY_RE = re.compile(
    r'<DataArray[^>]*Name\s*=\s*"([^"]+)"[^>]*>(.*?)</DataArray>',
    flags=re.DOTALL,
)
COORDINATE_ARRAY_NAMES = ("X_COORDINATES", "Y_COORDINATES", "Z_COORDINATES")


def read_index_lookup(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame(columns=["Index"])
    lookup = pd.read_csv(path, sep=r"\s+")
    lookup["Index"] = lookup["Index"].astype(int)
    return lookup


def read_ascii_vtr_index(path, dtype=np.int32):
    path = Path(path)
    text = path.read_text(encoding="utf-8")
    arrays = {}
    for name, body in VTR_DATA_ARRAY_RE.findall(text):
        name = name.strip()
        array_dtype = (
            np.float64 if name in COORDINATE_ARRAY_NAMES else np.float32
        )
        arrays[name] = np.fromstring(body, sep=" ", dtype=array_dtype)

    missing = [name for name in COORDINATE_ARRAY_NAMES if name not in arrays]
    if missing:
        raise ValueError(f"Missing coordinate arrays in {path.name}: {missing}")
    data_names = [
        name for name in arrays if name not in COORDINATE_ARRAY_NAMES
    ]
    if len(data_names) != 1:
        raise ValueError(
            f"Expected one data array in {path.name}, got {data_names}"
        )

    x, y, z = (arrays[name] for name in COORDINATE_ARRAY_NAMES)
    values = np.rint(arrays[data_names[0]]).astype(dtype, copy=False)
    expected = len(x) * len(y) * len(z)
    if values.size != expected:
        raise ValueError(
            f"{path.name}: expected {expected:,} values, got {values.size:,}"
        )
    return {
        "x": x,
        "y": y,
        "z": z,
        "values": values.reshape((len(z), len(y), len(x))),
    }


def sample_contact_points(contact_grid, contact_index, z_range_nm, max_points=3500):
    mask = contact_grid["values"] == int(contact_index)
    if z_range_nm is not None:
        z_min, z_max = z_range_nm
        z_mask = (
            (contact_grid["z"] >= z_min)
            & (contact_grid["z"] <= z_max)
        )
        mask = mask & z_mask[:, None, None]
    zz, yy, xx = np.nonzero(mask)
    if xx.size == 0:
        return None
    if xx.size > max_points:
        selection = np.linspace(0, xx.size - 1, max_points, dtype=int)
        zz, yy, xx = zz[selection], yy[selection], xx[selection]
    return contact_grid["x"][xx], contact_grid["y"][yy], contact_grid["z"][zz]

In [16]:
if RUN_AVAILABLE and STRUCTURE_DIR.exists():
    contact_lookup = read_index_lookup(STRUCTURE_DIR / "contact_indices.txt")
    contact_grid = read_ascii_vtr_index(
        STRUCTURE_DIR / "contacts.vtr",
        dtype=np.int16,
    )
    contact_name_to_index = dict(
        zip(contact_lookup["Contact"], contact_lookup["Index"])
    )
    colors_by_type = {
        "plunger": "#e41a1c",
        "barrier": "#ff7f00",
        "ohmic": "#984ea3",
        "screening": "#377eb8",
    }

    nextnano_structure_fig = go.Figure()
    for region in simulation_layout.patterned_regions:
        if region.name not in contact_name_to_index:
            continue
        sampled = sample_contact_points(
            contact_grid,
            contact_name_to_index[region.name],
            z_range_nm=(-120.0, 180.0),
        )
        if sampled is None:
            continue
        xs, ys, zs = sampled
        nextnano_structure_fig.add_trace(go.Scatter3d(
            x=xs,
            y=ys,
            z=zs,
            mode="markers",
            name=region.name,
            marker={
                "size": 2.8,
                "color": colors_by_type.get(region.gate_type, "#666666"),
                "opacity": 0.86,
            },
            hovertemplate=(
                f"{region.name}<br>x=%{{x:.1f}} nm"
                "<br>y=%{y:.1f} nm<br>z=%{z:.1f} nm<extra></extra>"
            ),
        ))

    nextnano_structure_fig.update_layout(
        title="Actual nextnano contact-index structure (2x3 array)",
        height=760,
        margin={"l": 0, "r": 0, "t": 48, "b": 0},
        scene={
            "xaxis_title": "x (nm)",
            "yaxis_title": "y (nm)",
            "zaxis_title": "z (nm)",
            "aspectmode": "manual",
            "aspectratio": {"x": 1.4, "y": 0.9, "z": 0.8},
            "camera": {
                "eye": {"x": 1.15, "y": -1.35, "z": 2.15},
                "projection": {"type": "orthographic"},
            },
        },
    )
    nextnano_structure_fig.show()
else:
    print("Run section 4 first to plot the actual nextnano Structure output.")

## 6. Quick diagnostic: integrated_hole_density

In [17]:
if RUN_AVAILABLE:
    integrated_path = Path(OUTPUT_RUN_DIR) / "integrated_density_hole.dat"
    if integrated_path.exists():
        integrated_hole_density = nnt.read_integrated_density_hole(
            integrated_path
        )
        display(integrated_hole_density)
        region_columns = nnt.integrated_density_region_columns(
            integrated_hole_density
        )
        if region_columns:
            display(integrated_hole_density[region_columns].describe())
    else:
        integrated_hole_density = None
        print("No integrated_density_hole.dat was written by this run.")
else:
    integrated_hole_density = None
    print("Run section 4 first to report integrated_hole_density.")

,OC_L_1_bias[V],B1_bias[V],P1_bias[V],B2_bias[V],P2_bias[V],B3_bias[V],P3_bias[V],B4_bias[V],OC_R_1_bias[V],OC_L_2_bias[V],...,B6_bias[V],P5_bias[V],B7_bias[V],P6_bias[V],B8_bias[V],OC_R_2_bias[V],Body_bias[V],remove_surface_charge_bias[V],zero_fermi_QW_bias[V],region_2[carriers]
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,-1.5,0,7561238.6


,region_2[carriers]
count,1.0
mean,7561238.6
std,NaN
min,7561238.6
25%,7561238.6
50%,7561238.6
75%,7561238.6
max,7561238.6


## 7. Chosen plane cuts: potential and hole density

In [18]:
PLANE_AXIS = "z"
PLANE_VALUE_NM = -2.0

if RUN_AVAILABLE:
    for quantity, variable in [
        ("potential", "Potential"),
        ("density_hole", "Hole_density"),
    ]:
        try:
            nnt.plot_bias_volume_slice(
                OUTPUT_RUN_DIR,
                quantity,
                bias=BIAS_INDEX,
                variable=variable,
                slice_axis=PLANE_AXIS,
                slice_value=PLANE_VALUE_NM,
                interactive=True,
                log10=False,
                title=f"{variable}: {PLANE_AXIS} = {PLANE_VALUE_NM:g} nm",
            )
        except (FileNotFoundError, ValueError) as error:
            print(f"{quantity}: unavailable in this run ({error})")
else:
    print("Run section 4 first, or select a compatible completed run.")

## 8. Row and column line cuts

The two x-directed cuts cross all three dots in each row. The three y-directed cuts cross the opposing dot pair in each column.

In [19]:
row_center_y_nm = [
    0.5 * two_d_array.device_y_size_nm,
    two_d_array.row_pitch_y_nm + 0.5 * two_d_array.device_y_size_nm,
]
column_center_x_nm = [
    elements_by_name[f"P{index}"]["center_x_nm"]
    for index in range(1, 4)
]
LINE_Z_NM = -2.0
LINE_CUT_SPECS = [
    {
        "label": f"row {index + 1}",
        "axis": "x",
        "fixed_coords": {"y": y_nm, "z": LINE_Z_NM},
    }
    for index, y_nm in enumerate(row_center_y_nm)
] + [
    {
        "label": f"column {index + 1}",
        "axis": "y",
        "fixed_coords": {"x": x_nm, "z": LINE_Z_NM},
    }
    for index, x_nm in enumerate(column_center_x_nm)
]

display(pd.DataFrame(LINE_CUT_SPECS))

if RUN_AVAILABLE:
    for quantity, variable in [
        ("potential", "Potential"),
        ("density_hole", "Hole_density"),
    ]:
        try:
            path = nnt.resolve_bias_output_file(
                OUTPUT_RUN_DIR,
                quantity,
                bias=BIAS_INDEX,
                preferred_extensions=("vtr",),
            )
            for spec in LINE_CUT_SPECS:
                nnt.plot_vtr_linecut(
                    path,
                    variable=variable,
                    axis=spec["axis"],
                    fixed_coords=spec["fixed_coords"],
                    interactive=True,
                    title=(
                        f"{variable}: {spec['label']}; "
                        f"fixed {spec['fixed_coords']}"
                    ),
                )
        except (FileNotFoundError, ValueError) as error:
            print(f"{quantity}: unavailable in this run ({error})")
else:
    print("Run section 4 first, or select a compatible completed run.")

,label,axis,fixed_coords
0,row 1,x,"{'y': 100.0, 'z': -2.0}"
1,row 2,x,"{'y': 300.0, 'z': -2.0}"
2,column 1,y,"{'x': -180.0, 'z': -2.0}"
3,column 2,y,"{'x': 0.0, 'z': -2.0}"
4,column 3,y,"{'x': 180.0, 'z': -2.0}"
